# Discovery Analysis
ESM-2 zero-shot mutation scoring + PED structural features → aggregation rate correlation

**28 disease mutations** across tau K18, α-synuclein, Aβ42

In [ ]:
# Setup
import os, sys
os.chdir('/content')
!rm -rf brain_idp_flow
!git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm scipy -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load targets (28 mutations)
from brain_idp_flow.targets import load_targets
targets = load_targets('configs/targets.yaml')

total_muts = sum(len(t.mutations) for t in targets.values())
for tid, t in targets.items():
    print(f'{tid}: {t.name} — {len(t.mutations)} mutations')
print(f'\nTotal: {total_muts} mutations')

In [ ]:
# Step 1: ESM-2 Log-Likelihood Ratio scoring (zero-shot, ~5 min)
from brain_idp_flow.analysis.esm2_llr import score_all_mutations

esm2_results = score_all_mutations(targets, device=device)
print(f'\nScored {len(esm2_results)} mutations')

In [ ]:
# Step 2: PED structural features (~10 min with download)
os.makedirs('data/ped', exist_ok=True)
from brain_idp_flow.analysis.ped_features import extract_all_ped_features

ped_results = extract_all_ped_features(targets, cache_dir='data/ped')
print(f'\nExtracted features for {len(ped_results)} mutations')

In [ ]:
# Step 3: Correlation analysis + figures
from brain_idp_flow.analysis.aggregation_predictor import run_correlation_analysis

summary = run_correlation_analysis(esm2_results, ped_results, output_dir='results')

# Show key results
print('\n' + '='*60)
print('KEY FINDINGS')
print('='*60)
for feat, info in sorted(summary['correlations'].items(), 
                          key=lambda x: x[1]['p_value']):
    if info['p_value'] < 0.05:
        print(f"SIGNIFICANT: {info['label']} — "
              f"ρ={info['spearman_rho']:.3f}, p={info['p_value']:.4f}")

In [ ]:
# Display figures
from IPython.display import Image, display

for img in sorted(os.listdir('results')):
    if img.endswith('.png'):
        print(f'\n--- {img} ---')
        display(Image(f'results/{img}'))

In [ ]:
# Save results + Drive backup
import json

with open('results/full_results.json', 'w') as f:
    json.dump({
        'esm2_results': esm2_results,
        'ped_results': ped_results,
        'summary': summary,
    }, f, indent=2, default=float)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    backup = '/content/drive/MyDrive/brain_idp_flow_discovery'
    shutil.copytree('results', backup, dirs_exist_ok=True)
    print(f'Backed up to {backup}')
except:
    print('Drive backup skipped')

print('\n=== DONE ===')